# 03 · Reconstruct 2023, validate independently, export for the console
Test year 2023 was never seen in training. Scores against **GLORYS** (the target), **HYCOM** (your GEE,
independent model) and **Argo** profiles (independent observations), for NIO / Bay of Bengal / Arabian Sea.

In [ ]:
# ── Setup: Drive + code + deps ─────────────────────────────────────────────
from google.colab import drive, userdata
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/OceanEmbed'          # data, checkpoints, outputs live here
CODE = f'{ROOT}/code'                               # upload the local OceanEmbed/oceanembed folder here
import os, sys; os.makedirs(ROOT, exist_ok=True)
assert os.path.exists(f'{CODE}/oceanembed/__init__.py'), 'Upload the oceanembed/ package to MyDrive/OceanEmbed/code/'
sys.path.insert(0, CODE)
!pip -q install copernicusmarine earthaccess argopy "xarray>=2024.6" zarr dask netcdf4 earthengine-api

In [ ]:
import torch, numpy as np, pandas as pd, xarray as xr, shutil
from oceanembed import config as C, dataset as D, train as TR, infer as I, argo as A, metrics as M
for s in ['inputs.zarr', 'target.zarr', 'hycom.zarr']:
    if os.path.exists(f'{ROOT}/{s}') and not os.path.exists(f'/content/{s}'): shutil.copytree(f'{ROOT}/{s}', f'/content/{s}')
S = D.load_stats(f'{ROOT}/stats.npz')
cfg = C.TrainConfig(window=15)
_, net = TR.new_models(cfg); TR.load_weights(net, f'{ROOT}/checkpoints/{cfg.arch}_w{cfg.window}/glorys_best.pt')
rec = I.reconstruct(net, '/content/inputs.zarr', S, *C.TEST, window=cfg.window)
rec.to_netcdf(f'{ROOT}/OceanEmbed_NIO_T_2023.nc')          # the PS deliverable

In [ ]:
# ── vs GLORYS and vs HYCOM, per depth and region ────────────────────────────
gl = xr.open_zarr('/content/target.zarr').thetao.sel(time=rec.time)
hy = xr.open_zarr('/content/hycom.zarr').thetao.sel(time=rec.time)
rows = []
for reg in C.REGIONS:
    mask = M.region_mask(reg)
    for name, ref in [('GLORYS', gl), ('HYCOM', hy)]:
        s = M.skill(np.where(mask, rec.thetao.values, np.nan), np.where(mask, ref.values, np.nan))
        rows += [dict(region=reg, vs=name, depth=d, **v) for d, v in s.items()]
skill = pd.DataFrame(rows); skill.to_csv(f'{ROOT}/skill_2023.csv', index=False)
skill.pivot_table(index='depth', columns=['region', 'vs'], values='rmse').round(3)

In [ ]:
# ── Independent Argo check: ours vs GLORYS vs HYCOM at the same profiles ────
prof = pd.read_parquet(f'{ROOT}/argo_2023.parquet')
m = A.match(A.match(A.match(prof, rec.thetao, 'ours'), gl, 'glorys'), hy, 'hycom')
tab = {n: A.score(m, n).set_index('depth')[['rmse', 'bias', 'r']] for n in ['ours', 'glorys', 'hycom']}
argo_skill = pd.concat(tab, axis=1); argo_skill.to_csv(f'{ROOT}/argo_skill_2023.csv'); argo_skill.round(3)

In [ ]:
# ── Is the predicted uncertainty honest? (fraction of Argo errors inside ±1σ should be ≈ 68 %) ──
sig = A.match(prof, rec.thetao_sigma, 'sig')
inside = [(np.abs(m[f'ours_T{d}'] - m[f'T{d}']) <= sig[f'sig_T{d}']).mean() for d in C.DEPTHS]
pd.Series(inside, index=C.DEPTHS, name='coverage@1σ').round(2)

In [ ]:
# ── Export for the web console → download the folder into OceanEmbed/web/public/data/ ──
days = pd.date_range('2023-05-01', '2023-05-31')   # e.g. the pre-monsoon / Cyclone Mocha window
I.export_web(rec.sel(time=days), f'{ROOT}/web_export', comparison=gl.sel(time=days).to_dataset(name='thetao'))
flat = argo_skill.copy(); flat.columns = [f'{a}_{b}' for a, b in flat.columns]   # ours_rmse, glorys_bias, ...
open(f'{ROOT}/web_export/skill.json', 'w').write(flat.reset_index().to_json(orient='records'))